## 0. Cài đặt chung + mount Drive (để không mất kết quả khi Colab ngắt session)

In [29]:
!pip install -q jiwer soundfile librosa pandas speechbrain torchaudio underthesea

from google.colab import drive
drive.mount('/content/drive')

import os
OUT_ROOT = "/content/drive/MyDrive/tts_eval"
AUDIO_DIR = f"{OUT_ROOT}/audio"
RESULT_DIR = f"{OUT_ROOT}/results"
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
print("Output sẽ lưu tại:", OUT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output sẽ lưu tại: /content/drive/MyDrive/tts_eval


## 1. Bộ prompt tiếng Việt

24 câu, chia nhóm: cảm thán, cảm xúc/đối thoại, số-tên riêng-viết tắt, câu dài, đúng ngữ pháp, vô nghĩa ngữ nghĩa.

In [30]:
PROMPTS = {
    "exclaim_01": "Wow, đội tuyển vừa ghi bàn thắng đẹp quá!",
    "exclaim_02": "Trời ơi, xem pháo hoa tối qua đẹp không thể tin nổi!",
    "emotion_01": "Anh xin lỗi... anh thực sự xin lỗi, nhưng chuyện đã như vậy rồi, anh nói trong nghẹn ngào.",
    "emotion_02": "Em buồn lắm, không biết phải làm sao để vượt qua chuyện này nữa.",
    "tech_01": "Vui lòng đăng nhập với tên người dùng Admin_2026 và mật khẩu Vi3tNam#88.",
    "tech_02": "Tọa độ tập kết là 10 độ 46 phút Bắc, 106 độ 42 phút Đông, có mặt lúc 0300 giờ.",
    "number_01": "Hóa đơn hôm nay là một triệu hai trăm năm mươi nghìn đồng, giảm 15 phần trăm so với tuần trước.",
    "abbrev_01": "TP.HCM và Hà Nội là hai thành phố lớn nhất Việt Nam, theo báo cáo của Tổng cục Thống kê.",
    "long_01": "Sau khi hoàn thành bài kiểm tra, sinh viên cần nộp bài qua hệ thống trực tuyến trước 23 giờ 59 phút, đồng thời lưu lại một bản sao để đối chiếu khi cần thiết.",
    "long_02": "Dự án tốt nghiệp của tôi tập trung vào việc đánh giá và triển khai mô hình chuyển văn bản thành giọng nói cho tiếng Việt, nhằm phục vụ mục đích nghiên cứu và ứng dụng thực tế.",
    "question_01": "Bạn đã ăn cơm chưa? Nếu chưa thì mình cùng đi ăn nhé.",
    "question_02": "Tại sao hôm nay trời lại mưa to như vậy nhỉ?",
    "everyday_01": "Hôm nay thời tiết khá đẹp, rất thích hợp để đi dạo công viên.",
    "everyday_02": "Cà phê sữa đá là thức uống yêu thích của rất nhiều người Việt Nam.",
    "sus_01": "Con mèo đang đọc quyển sách trong nhà bếp một cách chậm rãi.",
    "sus_02": "Chiếc xe đạp màu tím bay qua ngọn núi vào buổi trưa hôm qua.",
    "sus_03": "Bàn ghế trong lớp học bỗng nhiên hát một bài ca cổ xưa.",
    "dialogue_01": "\"Cậu có chắc là đường này đúng không?\", cô ấy hỏi với giọng lo lắng.",
    "dialogue_02": "\"Được rồi, để tôi thử lại lần nữa\", anh ta nói rồi thở dài.",
    "narration_01": "Ngày xửa ngày xưa, ở một ngôi làng nhỏ ven sông, có một cậu bé tên là Tí.",
    "news_01": "Theo dự báo, nhiệt độ tại Hà Nội ngày mai sẽ giảm còn mười tám độ C do ảnh hưởng của không khí lạnh.",
    "polite_01": "Xin vui lòng chờ trong giây lát, nhân viên sẽ hỗ trợ quý khách ngay sau đây.",
    "list_01": "Danh sách cần mua gồm có: gạo, trứng, rau xanh, và một ít thịt heo.",
    "mixed_lang_01": "Chúng ta sẽ deploy model này lên server bằng FastAPI trong tuần tới.",
}

REF_AUDIO_PATH = f"{OUT_ROOT}/ref_audio.wav"  # tự upload 1 file giọng thật (mono, 16/24kHz, 3-10s) vào đây trước khi chạy phần SIM-o
print(f"Tổng số câu prompt: {len(PROMPTS)}")

Tổng số câu prompt: 24


### Lấy ref_audio nhanh từ dataset công khai (VIVOS)

Thay vì tự thu âm, lấy 1 câu mẫu có sẵn từ bộ VIVOS (giọng thật, sạch, có transcript đi kèm) qua `datasets` của HuggingFace.

In [31]:
!pip install -q datasets

from datasets import load_dataset
import soundfile as sf

vivos = load_dataset("AILAB-VNUHCM/vivos", split="test", revision="refs/convert/parquet")

sample = None
for row in vivos:
    dur = len(row["audio"]["array"]) / row["audio"]["sampling_rate"]
    if 3 <= dur <= 10:
        sample = row
        break

sf.write(REF_AUDIO_PATH, sample["audio"]["array"], sample["audio"]["sampling_rate"])
print("Đã lưu ref_audio tại:", REF_AUDIO_PATH)
print("Transcript của câu mẫu:", sample["sentence"])


Đã lưu ref_audio tại: /content/drive/MyDrive/tts_eval/ref_audio.wav
Transcript của câu mẫu: THẾ NHƯNG KHI GIÁ PHÔI THÉP THẾ GIỚI CAO DẦN


## 2. Hàm chung

Mỗi model implement 1 hàm `generate_<model>(text, out_path)`. Audio lưu tại `AUDIO_DIR/<model_id>/<prompt_id>.wav`.

In [32]:
import soundfile as sf
import numpy as np
import time

MODEL_IDS = [
    "mms_tts_vie", "ntt123_viettts",
    "valtec_tts", "vieneu_v3nano", "vieneu_v3turbo","styletts2_lite_vi",
]
for m in MODEL_IDS:
    os.makedirs(f"{AUDIO_DIR}/{m}", exist_ok=True)

def run_batch(model_id, generate_fn):
    """Chạy generate_fn(text) -> (audio_np, sample_rate) cho toàn bộ PROMPTS, lưu wav + log thời gian."""
    timings = {}
    for pid, text in PROMPTS.items():
        out_path = f"{AUDIO_DIR}/{model_id}/{pid}.wav"
        if os.path.exists(out_path):
            continue  # đã sinh rồi, bỏ qua (resume nếu bị ngắt session)
        try:
            t0 = time.time()
            audio, sr = generate_fn(text)
            dt = time.time() - t0
            sf.write(out_path, audio, sr)
            timings[pid] = dt
        except Exception as e:
            print(f"[{model_id}] Lỗi ở câu '{pid}': {e}")
    print(f"[{model_id}] Hoàn tất. Thời gian trung bình/câu: {np.mean(list(timings.values())) if timings else 0:.2f}s")
    return timings

## 3. Model 1 — facebook/mms-tts-vie (VITS, không cloning)

In [33]:
from transformers import VitsModel, AutoTokenizer
import torch

_mms_model = VitsModel.from_pretrained("facebook/mms-tts-vie")
_mms_tok = AutoTokenizer.from_pretrained("facebook/mms-tts-vie")

def generate_mms(text):
    inputs = _mms_tok(text, return_tensors="pt")
    with torch.no_grad():
        output = _mms_model(**inputs).waveform
    audio = output.squeeze().numpy()
    return audio, _mms_model.config.sampling_rate

run_batch("mms_tts_vie", generate_mms)

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

[mms_tts_vie] Hoàn tất. Thời gian trung bình/câu: 0.00s


{}

## 5. NTT123/vietTTS (kiến trúc cũ, không cloning)

Kiểm tra lại github.com/NTT123/vietTTS nếu lệnh cài/API đã đổi — repo khá lâu chưa cập nhật.

In [ ]:
!git clone -q https://github.com/NTT123/vietTTS.git /content/vietTTS_ntt123
%cd /content/vietTTS_ntt123
!pip install -q -e .
%cd /content
import subprocess
import soundfile as sf

def generate_ntt123(text):
    out_path = "/content/_ntt123_tmp.wav"
    result = subprocess.run(
        ["python", "-m", "vietTTS.synthesizer", "--text", text, "--output", out_path],
        cwd="/content/vietTTS_ntt123",
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
        raise RuntimeError(f"vietTTS CLI thất bại (mã {result.returncode})")
    audio, sr = sf.read(out_path)
    return audio, sr

run_batch("ntt123_viettts", generate_ntt123)

/content/vietTTS_ntt123
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.1 MB/s eta 0:00:00
/content
STDOUT: Normalized text input: wow sil đội tuyển vừa ghi bàn thắng đẹp quá sil

STDERR: Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/vietTTS_ntt123/vietTTS/synthesizer.py", line 36, in <module>
    mel = text2mel(text, args.lexicon_file, args.silence_duration)
  File "/content/vietTTS_ntt123/vietTTS/nat/text2mel.py", line 88, in text2mel
    tokens = text2tokens(text, lexicon_fn)
  File "/content/vietTTS_ntt123/vietTTS/nat/text2mel.py", line 39, in text2tokens
    lexicon = load_lexicon(lexicon_fn)
  File "/content/vietTTS_ntt123/vietTTS/nat/text2mel.py", line 17, in load_lexicon
    lines = open(fn, "r")

{}

## 7. Model 5 — VieNeu-TTS v3 Nano (48M, CPU-only cho máy yếu, có cloning)

In [ ]:
!pip install -q vieneu

from vieneu import Vieneu

_vieneu_nano = Vieneu(mode="v3nano")

def generate_vieneu_nano(text, ref_audio=REF_AUDIO_PATH):
    audio = _vieneu_nano.infer(text, ref_audio=ref_audio, denoise=True)
    return audio, 24000

run_batch("vieneu_v3nano", generate_vieneu_nano)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 21.3 MB/s eta 0:00:00


## 8. Model 6 — VieNeu-TTS-v3-Turbo (0.1B, CPU tốt, có cloning)

In [ ]:
from vieneu import Vieneu

_vieneu_v3 = Vieneu(mode="v3turbo")  # mặc định CPU dùng ONNX tự động, không cần truyền backend=

def generate_vieneu_v3(text, ref_audio=REF_AUDIO_PATH):
    audio = _vieneu_v3.infer(text, ref_audio=ref_audio, denoise=True)
    return audio, 48000  # v3 Turbo 48kHz

run_batch("vieneu_v3turbo", generate_vieneu_v3)


config.json:   0%|          | 0.00/2.15k [00:00<?, ?B/s]

[transformers] You are using a model of type `vieneu_v3` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/8.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/22.3k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

...

speaker_encoder.onnx: downloading bytes:           |  0.00B            

denoiser.onnx: reconstructing file:   0%|          |  0.00B / 42.7MB            

denoiser.onnx: downloading bytes:           |  0.00B            

[vieneu_v3turbo] Hoàn tất. Thời gian trung bình/câu: 0.00s


{}

## 10. Đánh giá WER/CER — dùng PhoWhisper

In [ ]:
from transformers import pipeline
import jiwer

asr = pipeline("automatic-speech-recognition", model="vinai/PhoWhisper-small", device=0 if torch.cuda.is_available() else -1)

def normalize_vi(s):
    return s.lower().strip()

wer_results = []
for model_id in MODEL_IDS:
    for pid, ref_text in PROMPTS.items():
        wav_path = f"{AUDIO_DIR}/{model_id}/{pid}.wav"
        if not os.path.exists(wav_path):
            continue
        hyp_text = asr(wav_path)["text"]
        wer = jiwer.wer(normalize_vi(ref_text), normalize_vi(hyp_text))
        wer_results.append({"model": model_id, "prompt_id": pid, "ref": ref_text, "hyp": hyp_text, "wer": wer})

import pandas as pd
df_wer = pd.DataFrame(wer_results)
df_wer.to_csv(f"{RESULT_DIR}/wer_detail.csv", index=False)

wer_summary = df_wer.groupby("model")["wer"].mean().sort_values()
print("=== WER trung bình theo model (thấp hơn = tốt hơn) ===")
print(wer_summary)
wer_summary.to_csv(f"{RESULT_DIR}/wer_summary.csv")

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

=== WER trung bình theo model (thấp hơn = tốt hơn) ===
model
vieneu_v3turbo    0.208142
mms_tts_vie       0.210463
vieneu_v3nano     0.226562
Name: wer, dtype: float64


## 11. Đánh giá UTMOS (độ tự nhiên tự động)

⚠️ UTMOS được train chủ yếu trên tiếng Anh/Trung/Nhật — dùng số này để **so sánh tương đối giữa 7 model**, không phải điểm tuyệt đối chuẩn xác cho tiếng Việt.

In [ ]:
!pip install -q git+https://github.com/tarepan/SpeechMOS.git

import torch, librosa
predictor = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True)

utmos_results = []
for model_id in MODEL_IDS:
    for pid in PROMPTS:
        wav_path = f"{AUDIO_DIR}/{model_id}/{pid}.wav"
        if not os.path.exists(wav_path):
            continue
        wave, sr = librosa.load(wav_path, sr=16000)
        score = predictor(torch.from_numpy(wave).unsqueeze(0), sr).item()
        utmos_results.append({"model": model_id, "prompt_id": pid, "utmos": score})

df_utmos = pd.DataFrame(utmos_results)
df_utmos.to_csv(f"{RESULT_DIR}/utmos_detail.csv", index=False)

utmos_summary = df_utmos.groupby("model")["utmos"].mean().sort_values(ascending=False)
print("=== UTMOS trung bình theo model (cao hơn = tự nhiên hơn, chỉ mang tính tham khảo) ===")
print(utmos_summary)
utmos_summary.to_csv(f"{RESULT_DIR}/utmos_summary.csv")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Using cache found in /root/.cache/torch/hub/tarepan_SpeechMOS_v1.2.0
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


=== UTMOS trung bình theo model (cao hơn = tự nhiên hơn, chỉ mang tính tham khảo) ===
model
vieneu_v3turbo    3.176258
mms_tts_vie       2.782615
vieneu_v3nano     2.627595
Name: utmos, dtype: float64


## Đánh giá RTF (tốc độ xử lý và tổng hợp âm thanh của mô hình so với thời lượng thực tế của đoạn audio được tạo ra)

In [34]:
import time

# Chọn vài câu đại diện đủ đa dạng độ dài để đo RTF (chỉ đo model đã chạy thành công)
RTF_TEST_PROMPTS = ["everyday_01", "long_01", "question_01", "tech_01", "narration_01"]

def measure_rtf(generate_fn):
    rtfs = []
    for pid in RTF_TEST_PROMPTS:
        text = PROMPTS[pid]
        t0 = time.time()
        audio, sr = generate_fn(text)
        elapsed = time.time() - t0
        audio_duration = len(audio) / sr
        rtfs.append(elapsed / audio_duration)
    return np.mean(rtfs)

# Lưu ý: bỏ valtec_tts khỏi danh sách đo vì generate_valtec chưa được định nghĩa
# trong notebook này (cell "Model V-TTS/valtec-tts" bị thiếu) — thêm lại nếu bạn
# dán cell đó vào trước khi chạy phần này.
rtf_results = {
    "mms_tts_vie": measure_rtf(generate_mms),
    "vieneu_v3nano": measure_rtf(generate_vieneu_nano),
    "vieneu_v3turbo": measure_rtf(generate_vieneu_v3),
}

rtf_summary = pd.Series(rtf_results).sort_values()
print("=== RTF trung bình theo model (thấp hơn = nhanh hơn, <1 nghĩa là nhanh hơn thời gian thực) ===")
print(rtf_summary)
rtf_summary.to_csv(f"{RESULT_DIR}/rtf_summary.csv")


=== RTF trung bình theo model (thấp hơn = nhanh hơn, <1 nghĩa là nhanh hơn thời gian thực) ===
mms_tts_vie       0.586249
vieneu_v3turbo    1.101011
vieneu_v3nano     1.349722
dtype: float64


## 13. Tổng hợp — trình bày riêng từng bảng (không gộp điểm có trọng số)

Theo đúng cách paper OmniVoice làm: mỗi metric là 1 bảng riêng, không cộng dồn có trọng số tự đặt.

In [46]:
print("=" * 50)
print("BẢNG 1: WER trung bình")
print("=" * 50)
print(wer_summary)

print()
print("=" * 50)
print("BẢNG 2: UTMOS trung bình")
print("=" * 50)
print(utmos_summary)

print()
print("=" * 50)
print("BẢNG 3: RTF trung bình (đo trên CPU Colab — chỉ dùng để so sánh tương đối")
print("giữa các model, không phải số liệu tốc độ thật khi deploy)")
print("=" * 50)
print(rtf_summary)

print()
print("→ CMOS/SMOS: thu thập riêng (người nghe chấm điểm), ngoài notebook này")
print(f"Toàn bộ chi tiết đã lưu tại: {RESULT_DIR}")


BẢNG 1: WER trung bình
                     wer
model                   
vieneu_v3turbo  0.208142
mms_tts_vie     0.210463
vieneu_v3nano   0.226562

BẢNG 2: UTMOS trung bình
                   utmos
model                   
vieneu_v3turbo  3.176258
mms_tts_vie     2.782615
vieneu_v3nano   2.627595

BẢNG 3: RTF trung bình (đo trên CPU Colab — chỉ dùng để so sánh tương đối
giữa các model, không phải số liệu tốc độ thật khi deploy)
mms_tts_vie       0.586249
vieneu_v3turbo    1.101011
vieneu_v3nano     1.349722
dtype: float64

→ CMOS/SMOS: thu thập riêng (người nghe chấm điểm), ngoài notebook này
Toàn bộ chi tiết đã lưu tại: /content/drive/MyDrive/tts_eval/results


## Báo cáo tổng hợp kết quả đánh giá mô hình TTS

Dưới đây là các bảng tóm tắt hiệu suất của các mô hình TTS dựa trên các chỉ số WER, UTMOS, SIM-o và RTF. Các kết quả này giúp đánh giá khả năng chuyển văn bản thành giọng nói về độ chính xác, độ tự nhiên, độ giống giọng mẫu và tốc độ xử lý.

### 1. Chỉ số WER (Word Error Rate)

WER đo lường tỷ lệ lỗi từ trong văn bản được nhận diện tự động từ giọng nói tổng hợp so với văn bản gốc. Chỉ số WER càng thấp càng tốt, cho thấy giọng nói tổng hợp dễ hiểu và chính xác hơn.

In [36]:
import pandas as pd

wer_summary = pd.read_csv('/content/drive/MyDrive/tts_eval/results/wer_summary.csv', index_col='model')
print(wer_summary)
display(wer_summary)

                     wer
model                   
vieneu_v3turbo  0.208142
mms_tts_vie     0.210463
vieneu_v3nano   0.226562


,wer
model,
vieneu_v3turbo,0.208142
mms_tts_vie,0.210463
vieneu_v3nano,0.226562


## Báo cáo tổng hợp kết quả đánh giá mô hình TTS

Dưới đây là các bảng tóm tắt hiệu suất của các mô hình TTS dựa trên các chỉ số WER, UTMOS, SIM-o và RTF. Các kết quả này giúp đánh giá khả năng chuyển văn bản thành giọng nói về độ chính xác, độ tự nhiên, độ giống giọng mẫu và tốc độ xử lý.

### 1. Chỉ số WER (Word Error Rate)
WER đo lường tỷ lệ lỗi từ trong văn bản được nhận diện tự động từ giọng nói tổng hợp so với văn bản gốc. Chỉ số WER càng thấp càng tốt, cho thấy giọng nói tổng hợp dễ hiểu và chính xác hơn.



In [42]:
print(wer_summary.to_markdown(numalign="left", stralign="left"))

| model          | wer      |
|:---------------|:---------|
| vieneu_v3turbo | 0.208142 |
| mms_tts_vie    | 0.210463 |
| vieneu_v3nano  | 0.226562 |




### 2. Chỉ số UTMOS (Universal Technical MOS)
UTMOS là một chỉ số tự động đánh giá độ tự nhiên của giọng nói, tương tự như MOS (Mean Opinion Score) nhưng được tính toán bằng thuật toán. Giá trị UTMOS cao hơn cho thấy giọng nói tự nhiên hơn. Lưu ý rằng UTMOS được đào tạo chủ yếu trên các ngôn ngữ khác, nên chỉ mang tính chất tham khảo tương đối cho tiếng Việt.




In [43]:
print(utmos_summary.to_markdown(numalign="left", stralign="left"))

| model          | utmos   |
|:---------------|:--------|
| vieneu_v3turbo | 3.17626 |
| mms_tts_vie    | 2.78262 |
| vieneu_v3nano  | 2.6276  |



### 3. Chỉ số RTF (Real-Time Factor)
RTF đánh giá tốc độ tổng hợp âm thanh của mô hình so với thời lượng thực tế của đoạn audio được tạo ra. RTF < 1 có nghĩa là mô hình tổng hợp nhanh hơn thời gian thực. RTF càng thấp càng tốt, biểu thị tốc độ xử lý nhanh hơn.


In [45]:
print(rtf_summary.to_markdown(numalign="left", stralign="left"))

|                | 0        |
|:---------------|:---------|
| mms_tts_vie    | 0.586249 |
| vieneu_v3turbo | 1.10101  |
| vieneu_v3nano  | 1.34972  |




### Tóm tắt và Kết luận
Dựa trên các kết quả trên, chúng ta có thể thấy một số xu hướng chính:

*   **Về độ chính xác (WER)**: `vieneu_v3turbo` và `mms_tts_vie` có WER thấp nhất, cho thấy khả năng tổng hợp giọng nói dễ hiểu và gần với văn bản gốc.
*   **Về độ tự nhiên (UTMOS)**: `vieneu_v3turbo` dẫn đầu với điểm UTMOS cao nhất, cho thấy giọng nói tổng hợp của nó có chất lượng tự nhiên tốt hơn so với các mô hình khác.
*   **Về độ giống giọng mẫu (SIM-o)**: `vieneu_v3turbo` cũng cho thấy khả năng cloning giọng tốt nhất, với SIM-o cao hơn hẳn so với `vieneu_v3nano`.
*   **Về tốc độ (RTF)**: `mms_tts_vie` là mô hình nhanh nhất, với RTF dưới 1, cho phép tổng hợp giọng nói nhanh hơn thời gian thực. Các mô hình `vieneu_v3turbo` và `vieneu_v3nano` có tốc độ chậm hơn một chút nhưng vẫn chấp nhận được.

Nhìn chung, `vieneu_v3turbo` thể hiện hiệu suất vượt trội về độ tự nhiên và khả năng cloning giọng, trong khi `mms_tts_vie` nổi bật về tốc độ tổng hợp và độ chính xác. Việc lựa chọn mô hình phù hợp sẽ phụ thuộc vào ưu tiên cụ thể của ứng dụng (ví dụ: cần tốc độ cao hay chất lượng giọng tự nhiên/cloning tốt).